# Notebook 4: Fusion-Layer Attacks

Demonstrate all 9 fusion-layer attack types.

**Attacks:** Existence Suppression, Association Confusion, Cross-Sensor Consistency, False Track Injection, Sensor DoS, Track Merge Manipulation, Track Deletion, Track Swap, Stealthy Degradation

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, get_ground_truth, FusionAttackerDF
from attacks.fusion_attacks import FusionAttackType

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 4.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ground_truth = get_ground_truth(loader)

print('Sensors:', list(detections.keys()))
print('Targets:', list(ground_truth.keys()))

## 4.2 Initialize Attacker

In [ ]:
attacker = FusionAttackerDF()
print('Available attacks:', [a.name for a in FusionAttackType])

## 4.3 Run Fusion Attacks

In [ ]:
attacks = [
    FusionAttackType.EXISTENCE_SUPPRESSION,
    FusionAttackType.ASSOCIATION_CONFUSION,
    FusionAttackType.CROSS_SENSOR_CONSISTENCY,
    FusionAttackType.FALSE_TRACK_INJECTION,
    FusionAttackType.SENSOR_DOS,
    FusionAttackType.TRACK_MERGE_MANIPULATION,
    FusionAttackType.TRACK_DELETION,
    FusionAttackType.TRACK_SWAP,
    FusionAttackType.STEALTHY_DEGRADATION
]

results = {}
for attack in attacks:
    attacked = attacker.attack_scenario(detections.copy(), attack, ground_truth)
    results[attack.name] = attacked
    total = sum(len(df) for df in attacked.values())
    print(attack.name + ':', total, 'total detections')

## 4.4 Visualize Fusion Attack Impact

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

for idx, (attack_name, attacked_dict) in enumerate(results.items()):
    ax = axes[idx]
    
    for sid, df in detections.items():
        ax.scatter(df['x_piren'], df['y_piren'], s=2, alpha=0.2, c='blue')
    
    for sid, df in attacked_dict.items():
        ax.scatter(df['x_piren'], df['y_piren'], s=2, alpha=0.2, c='red')
    
    for tid, gt in ground_truth.items():
        ax.plot(gt['x_piren'], gt['y_piren'], 'k-', linewidth=1)
    
    ax.set_title(attack_name)
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Fusion-Layer Attacks', fontsize=14)
plt.tight_layout()
plt.show()

## 4.5 Detection Count Comparison

In [ ]:
benign_counts = {sid: len(df) for sid, df in detections.items()}

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(benign_counts))
width = 0.08

for idx, (attack_name, attacked_dict) in enumerate(results.items()):
    counts = [len(attacked_dict.get(sid, pd.DataFrame())) for sid in benign_counts.keys()]
    ax.bar(x + idx * width, counts, width, label=attack_name)

ax.set_xlabel('Sensor ID')
ax.set_ylabel('Detection Count')
ax.set_title('Detection Counts by Sensor After Fusion Attacks')
ax.set_xticks(x + width * 4)
ax.set_xticklabels(list(benign_counts.keys()))
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()